# NOTEBOOK 01: BUSINESS AND DATA UNDERSTANDING
## Computational Society Intelligence Platform (CSIP) — Modul MVP: BDI
**Kepatuhan SKKNI**:
- `[UK-01 Menentukan Objektif Bisnis]`
- `[UK-02 Menentukan Tujuan Teknis Data Science]`
- `[UK-03 Menelaah Data]`

---

### 1. Perumusan Masalah Bisnis (UK-01)
Tantangan bisnis utama dalam ekosistem marketplace e-commerce Olist:
1. **Rendahnya Retensi Pelanggan**: Mayoritas pelanggan hanya bertransaksi satu kali (~96.9% churn setelah order pertama).
2. **Friksi Biaya Pengiriman (Freight Ratio)**: Rasio ongkos kirim terhadap harga barang menjadi hambatan konversi keranjang multi-item.
3. **Fluktuasi Kualitas Pemenuhan Pesanan**: Keterlambatan melebihi estimasi secara tajam memicu ulasan negatif (<= 2 bintang).

### 2. Penentuan Tujuan Teknis Data Science (UK-02)
- **TT-01**: Integrasi 9 tabel multi-grain bersih tanpa time-travel leakage.
- **TT-02**: Segmentasi perilaku pelanggan (Silhouette >= 0.35).
- **TT-03**: Pengujian minimal 6 hipotesis (H1-H6) dengan kontrol FDR alpha=0.05.
- **TT-04**: Model klasifikasi repeat purchase dan ulasan negatif mengungguli baseline ladder B0-B4 (PR-AUC, ROC-AUC, Brier score).
- **TT-05**: Model regresi estimasi waktu pengiriman dengan reduksi MAE >= 10%.
- **TT-06**: Eksperimen Neural Network Tabular Embeddings & Text Deep Learning.
- **TT-07**: World Model transisi mingguan dengan prediction intervals (90%/95%).


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from csip.data.loader import OlistDataLoader, load_yaml_config
print("CSIP Modules successfully imported.")


CSIP Modules successfully imported.


### 3. Memuat dan Menelaah 9 Tabel Relasional Olist (UK-03)
Memuat tabel dengan fitur otomatis **memory downsizing** (float64 -> float32, int64 -> int16/int32, category) agar kompatibel dengan spesifikasi hardware laptop 8 GB RAM.


In [2]:
loader = OlistDataLoader(
    config_path=PROJECT_ROOT / "config" / "data.yaml",
    raw_data_dir=PROJECT_ROOT / "data" / "raw"
)

# Load all 9 tables
table_names = list(loader.tables_config.keys())
dataframes = {}

for name in table_names:
    dataframes[name] = loader.load_table(name, validate=True, use_cache=True)

# Ringkasan Profiling dan Penghematan Memori
summary_df = loader.build_validation_summary_df()
display(summary_df)


,table_name,passed,total_rows,duplicate_pk_rows,chronological_violations,initial_mb,downsized_mb,reduction_pct,warning_count,error_count
0,orders,True,99441,0,23,21.95,11.48,47.7,1,0
1,order_items,True,112650,0,0,18.37,14.83,19.3,0,0
2,order_payments,True,103886,0,0,8.11,4.85,40.1,0,0
3,order_reviews,True,99224,1603,0,17.84,13.68,23.3,1,0
4,customers,True,99441,0,0,11.03,9.80,11.2,0,0
5,sellers,True,3095,0,0,0.22,0.19,17.0,0,0
6,products,True,32951,0,0,3.73,2.17,41.9,0,0
7,geolocation,True,1000163,999120,0,50.12,30.09,40.0,1,0
8,category_translation,True,71,0,0,0.00,0.00,0.0,0,0


### 4. Evaluasi Penghematan Memori & Kesiapan Hardware (8 GB RAM)
Menghitung total reduksi penggunaan RAM hasil downcasting tipe data numerik dan kategori.


In [3]:
total_init = summary_df["initial_mb"].sum()
total_down = summary_df["downsized_mb"].sum()
total_saving = total_init - total_down
pct_saving = (total_saving / total_init) * 100

print(f"Total Memori Awal (Raw 64-bit): {total_init:.2f} MB")
print(f"Total Memori Hasil Downcasting : {total_down:.2f} MB")
print(f"Penghematan Memori             : {total_saving:.2f} MB ({pct_saving:.1f}% efisiensi)")


Total Memori Awal (Raw 64-bit): 131.37 MB
Total Memori Hasil Downcasting : 87.09 MB
Penghematan Memori             : 44.28 MB (33.7% efisiensi)


### 5. Ringkasan Relasi Kunci & Penutup
Seluruh 9 tabel berhasil ditelaah. Selanjutnya, Notebook 02 akan mengeksekusi validasi skema, pemeriksaan baris yatim (orphan records), deteksi anomali kronologi tanggal, dan isolasi kohort bebas right-censoring.
